# Risk & factor analytics with `Performance`

Every analytic in this notebook flows through a single `Performance` instance — construct it once from prices (or returns), then call methods for return / risk metrics, tail diagnostics, benchmark-relative views, rolling regressions, and basic factor models.


## Build a small panel

We simulate two correlated daily price series — a fund and a benchmark — over one trading year, then construct a single `Performance` for the panel.


In [ ]:
import datetime
import random

from finstack.analytics import Performance

random.seed(42)
n = 252

start = datetime.date(2024, 1, 2)
dates = []
d = start
for _ in range(n):
    while d.weekday() >= 5:
        d += datetime.timedelta(days=1)
    dates.append(d)
    d += datetime.timedelta(days=1)

bench_prices = [100.0]
fund_prices = [100.0]
for _ in range(n - 1):
    bench_r = random.gauss(0.0004, 0.01)
    fund_r = 0.00025 + 0.88 * bench_r + random.gauss(0.0, 0.0035)
    bench_prices.append(bench_prices[-1] * (1.0 + bench_r))
    fund_prices.append(fund_prices[-1] * (1.0 + fund_r))

perf = Performance.from_arrays(
    dates,
    [fund_prices, bench_prices],
    ["FUND", "BENCH"],
    benchmark_ticker="BENCH",
)
print("Tickers:", perf.ticker_names)
print("Bench idx:", perf.benchmark_idx)
print("Active dates:", len(perf.dates()))

## Single-factor benchmark regression

`Performance.beta()` returns OLS beta with standard error and a 95% CI per ticker; `greeks()` returns annualized alpha, beta, and R². Both use the panel's designated benchmark.


In [ ]:
betas = perf.beta()
gks = perf.greeks()

fund_beta = betas[0]
fund_gr = gks[0]
print(f"Beta: {fund_beta.beta:.4f} (SE {fund_beta.std_err:.4f})")
print(f"95% CI: [{fund_beta.ci_lower:.4f}, {fund_beta.ci_upper:.4f}]")
print(f"Greeks alpha (ann.): {fund_gr.alpha:.6f}")
print(f"Greeks beta: {fund_gr.beta:.4f}; R²: {fund_gr.r_squared:.4f}")

## Rolling greeks

`Performance.rolling_greeks(ticker_idx, window)` fits the single-index model on a moving window and returns time series of alpha and beta aligned to right-labelled window-end dates.


In [ ]:
rg = perf.rolling_greeks(0, window=63)
print(f"Rolling windows: {len(rg.alphas)}")
print(f"Last alpha: {rg.alphas[-1]:.6f}; last beta: {rg.betas[-1]:.4f}")

## VaR, ES, and tail diagnostics

`Performance` exposes historical, parametric, and Cornish–Fisher VaR plus expected shortfall, as well as higher moments (`skewness`, `kurtosis`) and the asymmetric `tail_ratio`.


In [ ]:
conf = 0.95
print(f"Historical VaR ({conf:.0%}): {perf.value_at_risk(conf)[0]:.6f}")
print(f"Parametric VaR ({conf:.0%}): {perf.parametric_var(conf)[0]:.6f}")
print(f"Cornish–Fisher VaR ({conf:.0%}): {perf.cornish_fisher_var(conf)[0]:.6f}")
print(f"Expected shortfall ({conf:.0%}): {perf.expected_shortfall(conf)[0]:.6f}")
print(f"Skewness: {perf.skewness()[0]:.4f}")
print(f"Kurtosis: {perf.kurtosis()[0]:.4f}")
print(f"Tail ratio: {perf.tail_ratio()[0]:.4f}")

## Benchmark-relative metrics

Up/down capture, capture ratio, tracking error, information ratio, R², and batting average — all per-ticker against the panel benchmark.


In [ ]:
print(f"Up capture: {perf.up_capture()[0]:.4f}")
print(f"Down capture: {perf.down_capture()[0]:.4f}")
print(f"Capture ratio: {perf.capture_ratio()[0]:.4f}")
print(f"Tracking error: {perf.tracking_error()[0]:.6f}")
print(f"Information ratio: {perf.information_ratio()[0]:.4f}")
print(f"R-squared: {perf.r_squared()[0]:.4f}")
print(f"Batting average: {perf.batting_average()[0]:.4f}")

## Drawdown episodes

`drawdown_details(ticker_idx, n=...)` returns the deepest drawdown episodes for a single ticker with start, valley, and recovery dates.


In [ ]:
print(f"Max drawdown (fund): {perf.max_drawdown()[0]:.2%}")
episodes = perf.drawdown_details(0, n=3)
for i, ep in enumerate(episodes, start=1):
    end = ep.end()
    end_s = end.isoformat() if end else "(ongoing)"
    print(
        f"  #{i} Start: {ep.start().isoformat()}, Valley: {ep.valley().isoformat()}, "
        f"End: {end_s}, Depth: {ep.max_drawdown:.2%}"
    )

## Multi-factor model

`Performance.multi_factor_greeks(ticker_idx, factor_returns)` fits a multivariate OLS against several aligned factor return series and returns alpha, betas, R², adjusted R², and residual volatility.


In [ ]:
active_dates = perf.dates()
n_active = len(active_dates)
random.seed(7)
smb_returns = [random.gauss(0.0, 0.005) for _ in range(n_active)]
hml_returns = [random.gauss(0.0, 0.004) for _ in range(n_active)]
bench_returns = perf.cumulative_returns()[perf.benchmark_idx]
bench_period_returns = [
    bench_returns[i] if i == 0 else (1.0 + bench_returns[i]) / (1.0 + bench_returns[i - 1]) - 1.0
    for i in range(n_active)
]

mf = perf.multi_factor_greeks(0, [bench_period_returns, smb_returns, hml_returns])
print(f"Alpha (ann.): {mf.alpha:.6f}")
print(f"Betas [mkt, size, value]: {[round(b, 4) for b in mf.betas]}")
print(f"R²: {mf.r_squared:.4f}; Adj R²: {mf.adjusted_r_squared:.4f}")
print(f"Residual vol: {mf.residual_vol:.6f}")

## Periodic returns

`lookback_returns` reports period-to-date returns (MTD / QTD / YTD, and FYTD when a fiscal year start month is provided). `rolling_returns(ticker_idx, window)` compounds returns over a rolling window for trailing 1Y / 3Y views.


In [ ]:
ref = perf.dates()[-1]
lb = perf.lookback_returns(ref, fiscal_year_start_month=4)
print(f"As of {ref.isoformat()}:")
print(f"  MTD: {lb.mtd[0]:.4f}")
print(f"  QTD: {lb.qtd[0]:.4f}")
print(f"  YTD: {lb.ytd[0]:.4f}")
if lb.fytd is not None:
    print(f"  FYTD (Apr): {lb.fytd[0]:.4f}")

roll_1y = perf.rolling_returns(0, window=252)
print(f"Trailing 1Y rolling returns: {len(roll_1y.values)} windows")
if roll_1y.values:
    print(f"  Last window end: {roll_1y.dates[-1].isoformat()}")
    print(f"  Last 1Y compounded return: {roll_1y.values[-1]:.4f}")

## Takeaways

- `Performance` is the single binding entry point: construct from prices (or returns) and reach every analytic as a method.
- Benchmark regressions (`beta`, `greeks`, `rolling_greeks`) and multi-factor models (`multi_factor_greeks`) cover the alpha / beta / R² questions.
- Tail risk (`value_at_risk`, `expected_shortfall`, parametric / Cornish–Fisher variants, higher moments, `tail_ratio`) and benchmark-relative metrics live on the same instance.
- Periodic and rolling views (`lookback_returns`, `rolling_returns`, `rolling_sharpe`, `rolling_volatility`) answer MTD / QTD / YTD / FYTD and trailing-window questions directly.
